# DICE FEST 2026: submisi dari artefak model tersimpan

Memuat model per fold dari `final/artifacts/models/` (checksum di `final/artifacts/SHA256SUMS`), membuat ulang prediksi OOF dan test setiap anggota tanpa training, lalu menjalankan tahap adopsi, blend, dan koreksi yang sama. Hasilnya harus identik byte dengan output notebook utama.

In [ ]:
import os, sys, platform, hashlib, json, subprocess, time
# ---- paths (edit DATA_DIR if the competition CSVs live elsewhere) ----
DATA_DIR = os.environ.get("DICEFEST_DATA_DIR", "data/raw")   # train.csv, test.csv, sample_submission.csv
ROOT = os.path.abspath("dicefest_nb")                         # embedded source code + work files
WORK = os.path.join(ROOT, "work")
OUT_CSV = os.path.join(ROOT, "submission.csv")
os.environ["DICEFEST_DATA_DIR"] = os.path.abspath(DATA_DIR)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # deterministic cuBLAS, before CUDA starts
os.makedirs(os.path.join(ROOT, "src"), exist_ok=True)
os.makedirs(os.path.join(ROOT, "configs"), exist_ok=True)
for f in ("train.csv", "test.csv", "sample_submission.csv"):
    assert os.path.exists(os.path.join(DATA_DIR, f)), f"missing {f} in {DATA_DIR}"
import numpy, pandas, sklearn, scipy, lightgbm, xgboost, torch, yaml, pyarrow
print("python", platform.python_version())
for m in (numpy, pandas, pyarrow, scipy, sklearn, lightgbm, xgboost, torch):
    print(f"{m.__name__:12s} {m.__version__}")
print("CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
EXPECTED = {"numpy": "2.5.4", "pandas": "3.0.6", "scikit-learn": "1.9.1", "lightgbm": "4.7.0",
            "xgboost": "3.4.1", "torch": "2.14.1+cu130"}
print("versions used for the reference run:", EXPECTED)
ARTIFACTS = os.environ.get("DICEFEST_ARTIFACTS", "final/artifacts")

## Kode sumber
Semua modul di bawah ini ditulis ke `dicefest_nb/src/` dan `dicefest_nb/configs/` lalu di-import. Isinya identik dengan `final/src/` dan `final/configs/` (dipakai juga oleh `final/run_all.py`).

In [ ]:
%%writefile dicefest_nb/src/common.py
"""Shared data loading, feature preparation and metric for DICE FEST 2026."""
import os

import numpy as np
import pandas as pd

SEED = 42

TARGET_COLS = [
    "target_macro_gas_kwh",
    "target_macro_elec_kwh",
    "target_eui_gas_kwh_per_sqm",
    "target_clean_building_ratio",
    "target_electric_heating_adoption",
]
SIGMAS = {
    "target_macro_gas_kwh": 151750.04,
    "target_macro_elec_kwh": 41725.37,
    "target_eui_gas_kwh_per_sqm": 40.65,
    "target_clean_building_ratio": 12.49,
    "target_electric_heating_adoption": 7.18,
}
HVAC = [f"hvac_dist_{i:02d}" for i in range(1, 11)]
ERA_MAP = {"High (<0%)": 0, "Medium (0-6%)": 1, "Low (6-18%)": 2, "Very Low (>18%)": 3}


def nrmse(y_true, y_pred, col):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)) / SIGMAS[col])


def holdout_split(n, frac=0.7, seed=7):
    """Fixed split of the train rows into P (models trained here) and H (holdout for fitting blend weights)."""
    perm = np.random.default_rng(seed).permutation(n)
    return np.sort(perm[: int(frac * n)]), np.sort(perm[int(frac * n):])


DATA_DIR = os.environ.get("DICEFEST_DATA_DIR", "data/raw")  # folder with train.csv, test.csv, sample_submission.csv
_CACHE = {}


def load(cache=None):
    """train / test frames read from DATA_DIR with the pyarrow CSV parser (bit-identical to the parquet cache used
    during development, see env/data_check.json). The argument is kept for API compatibility and ignored."""
    if "tr" not in _CACHE:
        _CACHE["tr"] = pd.read_csv(os.path.join(DATA_DIR, "train.csv"), engine="pyarrow")
        _CACHE["te"] = pd.read_csv(os.path.join(DATA_DIR, "test.csv"), engine="pyarrow")
    return _CACHE["tr"].copy(), _CACHE["te"].copy()


def load_sample():
    return pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"), engine="pyarrow")


def base_features(df, const_cols):
    """Numeric feature frame: encoded strings, id codes, and a few physics-motivated ratios."""
    X = df.drop(columns=[c for c in df.columns if c.startswith("target_")] + ["id"] + const_cols).copy()
    X["bld_dim_18"] = X["bld_dim_18"].map(ERA_MAP).astype("float32")
    X["socio_demog_016"] = (X["socio_demog_016"] == "Urban").astype("float32")
    X["zone_n"] = X.pop("zone_region_id").str[4:].astype("int32")
    X["sub_n"] = X.pop("subzone_id").str[4:].astype("int32")
    A = df["bld_dim_06"]
    X["hvac_rest"] = 100 - df[HVAC].sum(axis=1)
    X["area_per_meter"] = A / df["bld_dim_12"]
    X["area_per_epc"] = A / df["bld_dim_01"]
    X["gea_ratio"] = A / df["bld_dim_03"]
    X["a07_ratio"] = df["bld_dim_07"] / A
    X["a08_ratio"] = df["bld_dim_08"] / A
    X["a05_ratio"] = df["bld_dim_05"] / df["bld_dim_03"]
    X["a04_ratio"] = df["bld_dim_04"] / df["bld_dim_03"]
    X["s002_ratio"] = df["socio_demog_002"] / A
    X["meters_per_epc"] = df["bld_dim_12"] / df["bld_dim_01"]
    X["clean_ratio"] = df["bld_dim_02"] / df["bld_dim_01"]
    return X.astype("float32")


def constant_cols(tr):
    num = tr.select_dtypes("number")
    return [c for c in num.columns if num[c].nunique() <= 1]


In [ ]:
%%writefile dicefest_nb/src/te.py
"""Out-of-fold group target statistics (hierarchical spatial encodings)."""
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold


def group_stats(fit_keys, fit_vals, apply_keys):
    """Sum and count of `fit_vals` columns per key, looked up for `apply_keys`.

    fit_vals: 2-D array (n_fit, k). Returns (sums (n_apply, k), counts (n_apply,)).
    """
    df = pd.DataFrame(fit_vals)
    df["_k"] = fit_keys
    g = df.groupby("_k", sort=False)
    sums = g.sum()
    cnt = g.size()
    idx = pd.Index(sums.index).get_indexer(apply_keys)
    out_s = np.zeros((len(apply_keys), fit_vals.shape[1]))
    out_c = np.zeros(len(apply_keys))
    ok = idx >= 0
    out_s[ok] = sums.values[idx[ok]]
    out_c[ok] = cnt.values[idx[ok]]
    return out_s, out_c


def oof_group_stats(keys, vals, n_splits=5, seed=42):
    """K-fold out-of-fold sums/counts for every row of one set."""
    s = np.zeros((len(keys), vals.shape[1]))
    c = np.zeros(len(keys))
    for a, b in KFold(n_splits, shuffle=True, random_state=seed).split(keys):
        s[b], c[b] = group_stats(keys[a], vals[a], keys[b])
    return s, c


def window_stats(fit_pos, fit_vals, apply_pos, w, size):
    """Sums/counts over fit rows whose position lies in [p-w, p+w], excluding position p itself.

    Positions are the numeric subzone ids, which are ordered roughly geographically.
    """
    k = fit_vals.shape[1]
    S = np.zeros((size + 2 * w + 2, k))
    C = np.zeros(size + 2 * w + 2)
    np.add.at(S, fit_pos + w + 1, fit_vals)
    np.add.at(C, fit_pos + w + 1, 1.0)
    cS = np.cumsum(S, axis=0)
    cC = np.cumsum(C)
    hi = apply_pos + 2 * w + 1
    lo = apply_pos
    s = cS[hi] - cS[lo] - S[apply_pos + w + 1]
    c = cC[hi] - cC[lo] - C[apply_pos + w + 1]
    return s, c


def oof_window_stats(pos, vals, w, size, n_splits=5, seed=42):
    s = np.zeros((len(pos), vals.shape[1]))
    c = np.zeros(len(pos))
    for a, b in KFold(n_splits, shuffle=True, random_state=seed).split(pos):
        s[b], c[b] = window_stats(pos[a], vals[a], pos[b], w, size)
    return s, c


def stats_to_features(s, c, names, prefix, prior=None, k=0.0):
    """Turn sums/counts into (optionally shrunk) means plus a count column."""
    out = {f"{prefix}_cnt": c}
    for j, n in enumerate(names):
        if prior is None:
            m = np.where(c > 0, s[:, j] / np.maximum(c, 1), np.nan)
        else:
            m = (s[:, j] + k * prior[j]) / (c + k)
        out[f"{prefix}_{n}"] = m
    return pd.DataFrame(out)


In [ ]:
%%writefile dicefest_nb/src/train_lib.py
"""Cross-validated training harness: one call = one (model, target formulation) pair.

Every run writes out-of-fold predictions (oof/<name>.npy) and the fold-averaged test
prediction (pred/<name>.npy) on the *physical* scale of the modelled quantity, so runs
can be blended afterwards.
"""
import os
import sys
import time

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

sys.path.insert(0, os.path.dirname(__file__))
from common import SEED, TARGET_COLS, base_features, constant_cols, load, nrmse  # noqa: E402
from te import group_stats, oof_group_stats, oof_window_stats, stats_to_features, window_stats  # noqa: E402

N_FOLDS = 5


class Data:
    def __init__(self, cache="cache", n_folds=N_FOLDS):
        tr, te = load(cache)
        const = constant_cols(tr)
        self.X = base_features(tr, const)
        self.Xt = base_features(te, const)
        self.A, self.At = tr["bld_dim_06"].values, te["bld_dim_06"].values
        self.M, self.Mt = tr["bld_dim_12"].values, te["bld_dim_12"].values
        self.y = {c: tr[c].values for c in TARGET_COLS}
        self.sub, self.subt = tr["subzone_id"].values, te["subzone_id"].values
        self.zone, self.zonet = tr["zone_region_id"].values, te["zone_region_id"].values
        self.subn, self.subnt = self.X["sub_n"].values.astype(int), self.Xt["sub_n"].values.astype(int)
        self.subn_max = int(max(self.subn.max(), self.subnt.max()))
        self.ids = te["id"].values
        self.folds = list(KFold(n_folds, shuffle=True, random_state=SEED).split(self.X))
        y_gas, y_elec, y_eui = (self.y[c] for c in TARGET_COLS[:3])
        # quantities that get spatially encoded
        self.V = np.c_[y_eui, np.log(y_eui), y_elec / self.M, np.log(y_elec / self.M), y_elec / self.A,
                       y_gas / self.M, np.log(y_gas)]
        self.VN = ["eui", "leui", "epm", "lepm", "epa", "gpm", "lgas"]
        self.W = np.c_[np.log(self.A), np.log(self.M)]
        self.WN = ["lA", "lM"]
        # climate-grid cells: rows sharing all climate values lie in the same grid square (~15 rows each)
        cl = pd.concat([tr, te], ignore_index=True)
        n = len(tr)
        cells = [cl.groupby([f"env_climate_0{i}" for i in g], sort=False, dropna=False).ngroup().values
                 for g in ((1, 3, 5),)]  # climate 02/04 repeat this grid (with a 0.0 filler)
        self.cells, self.cellst = [c[:n] for c in cells], [c[n:] for c in cells]

    # ---- spatial encodings -------------------------------------------------
    def te_features(self, fit_idx, apply="oof", windows=(), cells=False, self_idx=None):
        """apply='oof' -> inner-OOF features for fit_idx; else a tuple of apply-row keys
        (subzone, zone, sub_n[, list of cell keys]).

        self_idx: when the apply rows are themselves part of fit_idx (leave-one-out use), their own
        values are removed from the subzone/zone statistics (windows and cells already skip the own subzone)."""
        vals = np.c_[self.V, self.W]
        f = []
        if isinstance(apply, str) and apply == "oof":
            s, c = oof_group_stats(self.sub[fit_idx], vals[fit_idx])
            sz, cz = oof_group_stats(self.zone[fit_idx], self.V[fit_idx])
            f += [stats_to_features(s, c, self.VN + self.WN, "sz"), stats_to_features(sz, cz, self.VN, "zn")]
            for w in windows:
                sw, cw = oof_window_stats(self.subn[fit_idx], self.V[fit_idx], w, self.subn_max)
                f.append(stats_to_features(sw, cw, self.VN, f"w{w}"))
            if cells:  # cell statistics excluding rows of the own subzone
                for j, cell in enumerate(self.cells):
                    key = cell[fit_idx].astype(np.int64)
                    s1, c1 = oof_group_stats(key, self.V[fit_idx])
                    s2, c2 = oof_group_stats(key * 10 ** 6 + self.subn[fit_idx], self.V[fit_idx])
                    f.append(stats_to_features(s1 - s2, c1 - c2, self.VN, f"c{j}x"))
        else:
            keys_sub, keys_zone, pos = apply[:3]
            s, c = group_stats(self.sub[fit_idx], vals[fit_idx], keys_sub)
            sz, cz = group_stats(self.zone[fit_idx], self.V[fit_idx], keys_zone)
            if self_idx is not None:
                s, c = s - vals[self_idx], c - 1
                sz, cz = sz - self.V[self_idx], cz - 1
            f += [stats_to_features(s, c, self.VN + self.WN, "sz"), stats_to_features(sz, cz, self.VN, "zn")]
            for w in windows:
                sw, cw = window_stats(self.subn[fit_idx], self.V[fit_idx], pos, w, self.subn_max)
                f.append(stats_to_features(sw, cw, self.VN, f"w{w}"))
            if cells:
                for j, (cell, ka) in enumerate(zip(self.cells, apply[3])):
                    key, ka = cell[fit_idx].astype(np.int64), ka.astype(np.int64)
                    s1, c1 = group_stats(key, self.V[fit_idx], ka)
                    s2, c2 = group_stats(key * 10 ** 6 + self.subn[fit_idx], self.V[fit_idx], ka * 10 ** 6 + pos)
                    f.append(stats_to_features(s1 - s2, c1 - c2, self.VN, f"c{j}x"))
        return pd.concat(f, axis=1).astype("float32")

    def te_apply(self, idx=None):
        """Apply-row keys for te_features: train rows idx, or the test set when idx is None."""
        if idx is None:
            return self.subt, self.zonet, self.subnt, self.cellst
        return self.sub[idx], self.zone[idx], self.subn[idx], [c[idx] for c in self.cells]

    # ---- FB07: kNN target encodings in the building-stock space ------------------------------------------
    KNN_CONFIGS = ((30, 20, 0.0), (100, 20, 4.0), (300, 50, 0.0))  # (sub_n window W, k, spatial distance weight)

    def knn_space(self):
        """standardised building-stock space of train rows (self.X) and apply rows (self.Xt); cached"""
        if getattr(self, "_kz", None) is None:
            def raw(X):
                A, M, E = X["bld_dim_06"].values, X["bld_dim_12"].values.astype(float), X["bld_dim_01"].values.astype(float)
                era = X[[f"bld_era_{i:02d}" for i in range(9, 16)]].values / 100
                return np.column_stack([np.log(A / M), np.log(E / M), np.log(M), era, X["bld_dim_02"].values / E,
                                        np.log1p(X["bld_dim_17"].values), np.log(X["bld_dim_03"].values / A)])
            Z, Zt = raw(self.X), raw(self.Xt)
            mu, sd = Z.mean(0), Z.std(0)
            self._kz = ((Z - mu) / sd, (Zt - mu) / sd)
        return self._kz

    def knn_features(self, fit_idx, apply="oof", inner=5):
        """apply='oof' -> inner 5-fold cross-fitted features for fit_idx rows; ('train', idx) -> train rows idx;
        'test' -> apply rows (self.Xt). Values: log EUI, log elec / meter, log gas / meter of fit rows.
        Results are cached in self.knn_cache_dir (if set), keyed by a hash of the inputs (identical either way)."""
        import hashlib
        cdir = getattr(self, "knn_cache_dir", None)
        if cdir:
            h = hashlib.sha256()
            for arr in (np.asarray(fit_idx), self.subn, self.subnt, self.V[:, [1, 3, 5]]):
                h.update(np.ascontiguousarray(arr).tobytes())
            h.update(repr((apply if isinstance(apply, str) else ("train",), inner, self.KNN_CONFIGS)).encode())
            if not isinstance(apply, str):
                h.update(np.ascontiguousarray(apply[1]).tobytes())
            path = os.path.join(cdir, h.hexdigest()[:32] + ".parquet")
            if os.path.exists(path):
                return pd.read_parquet(path)
            F = self._knn_features(fit_idx, apply, inner)
            os.makedirs(cdir, exist_ok=True)
            tmp = f"{path}.{os.getpid()}.tmp"
            F.to_parquet(tmp, index=False)
            os.replace(tmp, path)
            return F
        return self._knn_features(fit_idx, apply, inner)

    def _knn_features(self, fit_idx, apply="oof", inner=5):
        from features_knn import knn_stats
        Z, Zt = self.knn_space()
        V = np.c_[np.log(self.V[:, 0]), self.V[:, 3], np.log(self.V[:, 5])]  # leui, lepm, lgpm

        def feats(f, Za, sa):
            cols = {}
            for W, k, sp in self.KNN_CONFIGS:
                v, c, dd = knn_stats(Z[f], self.subn[f], V[f], Za, sa, W, k, sp)
                tag = f"knn_w{W}_k{k}" + ("s" if sp else "")
                for j, nm in enumerate(("leui", "lepm", "lgpm")):
                    cols[f"{tag}_{nm}"] = v[:, j]
                cols[f"{tag}_cnt"] = c
                cols[f"{tag}_dist"] = dd
            return pd.DataFrame(cols).astype("float32")

        if isinstance(apply, str) and apply == "oof":
            parts, idx = [], []
            for ia, ib in KFold(inner, shuffle=True, random_state=SEED).split(fit_idx):
                parts.append(feats(fit_idx[ia], Z[fit_idx[ib]], self.subn[fit_idx[ib]]))
                idx.append(ib)
            F = pd.concat(parts, ignore_index=True)
            return F.iloc[np.argsort(np.concatenate(idx), kind="stable")].reset_index(drop=True)
        if isinstance(apply, str) and apply == "test":
            return feats(fit_idx, Zt, self.subnt)
        return feats(fit_idx, Z[apply[1]], self.subn[apply[1]])

    def agg_features(self, cache="cache"):
        """Subzone / zone aggregates over train and test rows: number of rows, log totals and row shares of
        meters, floor area and EPC count, mean log area per meter. Built from features only (no targets).
        Returns (train frame, test frame) for run(extra=..., extra_t=...)."""
        tr, te = load(cache)
        allr = pd.concat([tr, te], ignore_index=True)
        F = pd.DataFrame(index=allr.index)
        for gname in ("subzone_id", "zone_region_id"):
            g = allr.groupby(gname)
            t = gname[:3]
            F[f"{t}_cnt"] = g["id"].transform("size")
            for c, nm in (("bld_dim_12", "M"), ("bld_dim_06", "A"), ("bld_dim_01", "E")):
                s = g[c].transform("sum")
                F[f"{t}_sum{nm}"] = np.log1p(s)
                F[f"{t}_share{nm}"] = allr[c] / s
            F[f"{t}_lApM_mean"] = np.log(allr["bld_dim_06"] / allr["bld_dim_12"]).groupby(allr[gname]).transform("mean")
        F = F.astype("float32")
        n = len(tr)
        return F.iloc[:n].reset_index(drop=True), F.iloc[n:].reset_index(drop=True)


# ---- target formulations ------------------------------------------------------
# kind -> (training target, sample weight, multiplier back to modelled physical quantity)
def target_spec(d, kind, test=False):
    A = d.At if test else d.A
    M = d.Mt if test else d.M
    if kind.startswith(("gas_per:", "elec_per:", "lgas_per:", "lelec_per:")):
        # ratio formulations: target / D(col), weight D^2 for linear kinds (loss == target-scale MSE)
        fam, col = kind.split(":")
        D = (d.Xt if test else d.X)[col].values.astype(float)
        to_phys = D / A if fam in ("gas_per", "lgas_per") else D  # gas family stores EUI
        if test:
            return None, None, to_phys
        tgt = d.y[TARGET_COLS[0]] if fam in ("gas_per", "lgas_per") else d.y[TARGET_COLS[1]]
        if fam.startswith("l"):
            return np.log(tgt / D), None, to_phys
        return tgt / D, D ** 2 / np.mean(D ** 2), to_phys
    if test:
        return None, None, {"eui": 1.0, "gas": 1.0 / A, "gpm": M / A, "leui": 1.0,
                            "elec": 1.0, "epm": M, "epa": A, "lepm": M}.get(kind)
    y_gas, y_elec, y_eui = (d.y[c] for c in TARGET_COLS[:3])
    if kind == "eui":   # model EUI directly
        return y_eui, None, 1.0
    if kind == "gas":   # model gas, convert to EUI
        return y_gas, None, 1.0 / A
    if kind == "gpm":   # gas per meter, weighted so loss equals gas MSE
        return y_gas / M, M ** 2 / np.mean(M ** 2), M / A
    if kind == "leui":  # log EUI; prediction is exp()-ed (blend weights absorb the retransform bias)
        return np.log(y_eui), None, 1.0
    if kind == "elec":
        return y_elec, None, 1.0
    if kind == "lepm":
        return np.log(y_elec / M), None, M
    if kind == "epm":
        return y_elec / M, M ** 2 / np.mean(M ** 2), M
    if kind == "epa":
        return y_elec / A, A ** 2 / np.mean(A ** 2), A
    raise ValueError(kind)


# ---- model back-ends ------------------------------------------------------------
def fit_xgb(params, Xa, ya, wa, Xb, yb, wb, Xt, save_path=None):
    import xgboost as xgb
    p = dict(params)
    n = p.pop("n_estimators", 10000)
    es = p.pop("early_stopping_rounds", 200)
    p.setdefault("device", "cuda")
    p.setdefault("tree_method", "hist")
    p.setdefault("seed", SEED)
    mb = p.get("max_bin", 256)  # the quantile sketch must use the booster's bin count
    da = xgb.QuantileDMatrix(Xa, ya, weight=wa, max_bin=mb)
    db = xgb.QuantileDMatrix(Xb, yb, weight=wb, ref=da, max_bin=mb)
    m = xgb.train(p, da, n, evals=[(db, "va")], early_stopping_rounds=es, verbose_eval=False)
    rng = (0, m.best_iteration + 1)
    if save_path:
        m.save_model(save_path + ".ubj")
    return m.inplace_predict(Xb, iteration_range=rng), m.inplace_predict(Xt, iteration_range=rng), m.best_iteration


def fit_lgb(params, Xa, ya, wa, Xb, yb, wb, Xt, save_path=None):
    import lightgbm as lgb
    p = dict(objective="regression", seed=SEED, deterministic=True, force_row_wise=True, num_threads=16, verbose=-1)
    p.update(params)
    n = p.pop("n_estimators", 10000)
    es = p.pop("early_stopping_rounds", 200)
    da = lgb.Dataset(Xa, ya, weight=wa, free_raw_data=False)
    db = lgb.Dataset(Xb, yb, weight=wb, reference=da)
    m = lgb.train(p, da, n, valid_sets=[db], callbacks=[lgb.early_stopping(es, verbose=False)])
    if save_path:
        m.save_model(save_path + ".txt", num_iteration=m.best_iteration)
    return m.predict(Xb, num_iteration=m.best_iteration), m.predict(Xt, num_iteration=m.best_iteration), m.best_iteration


BACKENDS = {"xgb": fit_xgb, "lgb": fit_lgb}
LOG_KINDS = ("leui", "lepm")


def is_log_kind(kind):
    return kind in LOG_KINDS or kind.startswith(("lgas_per:", "lelec_per:"))


def is_gas_kind(kind):
    return kind in ("eui", "gas", "gpm", "leui") or kind.startswith(("gas_per:", "lgas_per:"))


def run(d, name, backend, kind, params, use_te=False, windows=(), drop=(), out_dir=".", verbose=True,
        wmul=None, extra=None, extra_t=None, cells=False, te_full=False, model_dir=None, knn=False):
    """K-fold CV; saves OOF and fold-averaged test predictions of the modelled physical quantity.

    The saved quantity is EUI for gas-family kinds and elec for elec-family kinds.
    te_full: encode validation rows from all other train rows (leave-one-out) and test rows from all
    train rows, instead of from the fold's fitting part only (training rows keep inner-OOF encodings).
    """
    fit = BACKENDS[backend]
    y, w, mult = target_spec(d, kind)
    mult_t = target_spec(d, kind, test=True)[2]
    if wmul is not None:  # extra per-row training weight (e.g. inverse noise variance)
        w = wmul / wmul.mean() if w is None else w * wmul / np.mean(w * wmul)
    oof = np.zeros(len(d.X))
    pt = np.zeros(len(d.Xt))
    its = []
    t0 = time.time()
    for k, (a, b) in enumerate(d.folds):
        Xa, Xb, Xt = d.X.iloc[a], d.X.iloc[b], d.Xt
        if extra is not None:  # additional precomputed feature frames (train rows / test rows)
            Xa = pd.concat([Xa.reset_index(drop=True), extra.iloc[a].reset_index(drop=True)], axis=1)
            Xb = pd.concat([Xb.reset_index(drop=True), extra.iloc[b].reset_index(drop=True)], axis=1)
            Xt = pd.concat([Xt.reset_index(drop=True), extra_t.reset_index(drop=True)], axis=1)
        if use_te:
            Xa = pd.concat([Xa.reset_index(drop=True), d.te_features(a, "oof", windows, cells)], axis=1)
            if te_full:
                if k == 0:
                    allr = np.arange(len(d.X))
                    Tt_full = d.te_features(allr, d.te_apply(), windows, cells)
                Tb = d.te_features(allr, d.te_apply(b), windows, cells, self_idx=b)
                Tt = Tt_full
            else:
                Tb = d.te_features(a, d.te_apply(b), windows, cells)
                Tt = d.te_features(a, d.te_apply(), windows, cells)
            Xb = pd.concat([Xb.reset_index(drop=True), Tb], axis=1)
            Xt = pd.concat([Xt.reset_index(drop=True), Tt], axis=1)
        if knn:  # FB07 kNN target encodings, cross-fitted like the target encodings
            Xa = pd.concat([Xa.reset_index(drop=True), d.knn_features(a, "oof")], axis=1)
            Xb = pd.concat([Xb.reset_index(drop=True), d.knn_features(a, ("train", b))], axis=1)
            Xt = pd.concat([Xt.reset_index(drop=True), d.knn_features(a, "test")], axis=1)
        if drop:
            Xa, Xb, Xt = (z.drop(columns=list(drop)) for z in (Xa, Xb, Xt))
        sp = None
        if model_dir:
            os.makedirs(model_dir, exist_ok=True)
            sp = f"{model_dir}/fold{k}"
        pb, ptk, it = fit(params, Xa, y[a], None if w is None else w[a], Xb, y[b], None if w is None else w[b], Xt,
                          save_path=sp)
        if is_log_kind(kind):
            pb, ptk = np.exp(pb), np.exp(ptk)
        oof[b] = pb * (mult if np.isscalar(mult) else mult[b])
        pt += ptk * mult_t / len(d.folds)
        its.append(it)
        if verbose:
            print(f"  [{name}] fold {k} it={it} {time.time() - t0:.0f}s", flush=True)
    os.makedirs(f"{out_dir}/oof", exist_ok=True)
    os.makedirs(f"{out_dir}/pred", exist_ok=True)
    np.save(f"{out_dir}/oof/{name}.npy", oof)
    np.save(f"{out_dir}/pred/{name}.npy", pt)
    if is_gas_kind(kind):
        msg = (f"eui={nrmse(d.y[TARGET_COLS[2]], oof, TARGET_COLS[2]):.5f} "
               f"gas={nrmse(d.y[TARGET_COLS[0]], oof * d.A, TARGET_COLS[0]):.5f}")
    else:
        msg = f"elec={nrmse(d.y[TARGET_COLS[1]], oof, TARGET_COLS[1]):.5f}"
    print(f"[{name}] {msg} iters={its} time={time.time() - t0:.0f}s", flush=True)
    return oof, pt


In [ ]:
%%writefile dicefest_nb/src/nn.py
"""Deterministic MLP regressor on the same folds as the GBMs (for ensemble diversity)."""
import os
import sys
import time

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import numpy as np
import pandas as pd
import torch
from torch import nn

sys.path.insert(0, os.path.dirname(__file__))
from common import SEED, TARGET_COLS, nrmse  # noqa: E402
from train_lib import is_gas_kind, is_log_kind, target_spec  # noqa: E402


def set_seed(seed=SEED):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)


def preprocess(X, Xt):
    """log1p on non-negative skewed columns, standardise with train statistics, NaN -> 0 plus flags."""
    X = X.copy()
    Xt = Xt.copy()
    for c in X.columns:
        v = X[c].values
        if np.nanmin(v) >= 0 and np.nanmax(v) > 0 and (np.nanmean(v) > 0) and \
                (np.nanpercentile(v, 99) > 5 * max(np.nanmedian(v), 1e-6)):
            X[c] = np.log1p(X[c])
            Xt[c] = np.log1p(Xt[c])
    nan_cols = [c for c in X.columns if X[c].isna().any() or Xt[c].isna().any()]
    mu = X.mean()
    sd = X.std().replace(0, 1)
    Z = ((X - mu) / sd).clip(-6, 6)
    Zt = ((Xt - mu) / sd).clip(-6, 6)
    Z = pd.concat([Z, X[nan_cols].isna().astype("float32").add_suffix("_nan")], axis=1)
    Zt = pd.concat([Zt, Xt[nan_cols].isna().astype("float32").add_suffix("_nan")], axis=1)
    return Z.fillna(0).values.astype("float32"), Zt.fillna(0).values.astype("float32")


class PLR(nn.Module):
    """Periodic embeddings of numerical features (Gorishniy et al. 2022): per feature
    x -> ReLU(Linear([sin(2 pi c x), cos(2 pi c x)])) with trainable frequencies c."""

    def __init__(self, n_feat, k=24, sigma=0.05, d=24):
        super().__init__()
        self.c = nn.Parameter(torch.randn(n_feat, k) * sigma)
        bound = 1 / np.sqrt(2 * k)
        self.w = nn.Parameter(torch.empty(n_feat, 2 * k, d).uniform_(-bound, bound))
        self.b = nn.Parameter(torch.empty(n_feat, d).uniform_(-bound, bound))

    def forward(self, x):
        v = 2 * np.pi * self.c[None] * x[..., None]
        z = torch.cat([torch.sin(v), torch.cos(v)], -1)
        return torch.relu(torch.einsum("bfk,fkd->bfd", z, self.w) + self.b[None]).flatten(1)


class MLP(nn.Module):
    """emb=None: plain MLP on the inputs; emb=dict(k, sigma, d): PLR embeddings of the non-binary columns
    (cont = boolean mask over input columns), binary columns passed through.
    ens=k > 0: TabM-mini (Gorishniy et al. 2024) - k members share the backbone; each member sees the input
    times its own random-sign vector and has its own output head. forward then returns (batch, k).
    n_out > 1 (plain head only): multi-task output, forward returns (batch, n_out)."""

    def __init__(self, d_in, hidden=(512, 256, 128), drop=0.1, emb=None, cont=None, ens=0, n_out=1):
        super().__init__()
        if emb is not None:
            self.register_buffer("ci", torch.tensor(np.flatnonzero(cont)))
            self.register_buffer("bi", torch.tensor(np.flatnonzero(~cont)))
            self.emb = PLR(len(self.ci), **emb)
            d_in = len(self.ci) * emb.get("d", 24) + len(self.bi)
        layers = []
        d = d_in
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(drop)]
            d = h
        self.ens = ens
        if ens:
            self.r = nn.Parameter(torch.randint(0, 2, (ens, d_in)).float() * 2 - 1)
            bound = 1 / np.sqrt(d)
            self.head_w = nn.Parameter(torch.empty(ens, d).uniform_(-bound, bound))
            self.head_b = nn.Parameter(torch.empty(ens).uniform_(-bound, bound))
        else:
            layers.append(nn.Linear(d, n_out))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        if hasattr(self, "emb"):
            x = torch.cat([self.emb(x[:, self.ci]), x[:, self.bi]], 1)
        if self.ens:
            h = self.net((x[:, None, :] * self.r[None]).flatten(0, 1)).view(len(x), self.ens, -1)
            return torch.einsum("bkd,kd->bk", h, self.head_w) + self.head_b
        return self.net(x).squeeze(-1)


def predict_model(model, Xs, emb, ens, mu, sd):
    """eval-mode prediction in fixed chunks (chunk sizes are part of the numerics), back on the target scale"""
    model.eval()
    with torch.no_grad():
        ch = 65536 if emb is None else 8192  # PLR activations are ~50x wider
        if ens:
            ch = 1024
        out = torch.cat([model(Xs[i:i + ch]) for i in range(0, len(Xs), ch)])
        if ens:
            out = out.mean(1)
    model.train()
    return out.cpu().numpy() * sd + mu


def fit_fold(Za, ya, wa, Zb, yb, wb, Zt, epochs=30, bs=4096, lr=2e-3, wd=1e-5, hidden=(512, 256, 128),
             drop=0.1, dev="cuda", fixed_ep=None, emb=None, order_seed=SEED, ens=0, save_path=None):
    """fixed_ep=None keeps the epoch with the best validation score (optimistic OOF);
    fixed_ep=k stops after epoch k of the same schedule without looking at the validation rows.
    emb: PLR embedding settings (see MLP), None for the plain MLP.
    order_seed: seed of the mini-batch order (the default SEED gives every seed and fold the same order).
    ens: TabM-mini members (see MLP); the loss averages the members' losses, predictions average the members.
    ya / wa of shape (n, t): multi-task fit of t targets with per-task weights (needs fixed_ep)."""
    multi = ya.ndim == 2
    mu, sd = (ya.mean(0), ya.std(0)) if multi else (ya.mean(), ya.std())
    g = torch.Generator(device="cpu").manual_seed(order_seed)
    cont = None if emb is None else np.array([len(np.unique(Za[:, j])) > 2 for j in range(Za.shape[1])])
    model = MLP(Za.shape[1], hidden, drop, emb, cont, ens, n_out=ya.shape[1] if multi else 1).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    steps = epochs * int(np.ceil(len(Za) / bs))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    Xa = torch.tensor(Za, device=dev)
    Ya = torch.tensor((ya - mu) / sd, dtype=torch.float32, device=dev)
    Wa = torch.tensor(np.ones(ya.shape) if wa is None else wa, dtype=torch.float32, device=dev)
    Xb = torch.tensor(Zb, device=dev)
    Xt = torch.tensor(Zt, device=dev)
    wbn = np.ones(len(yb)) if wb is None else wb

    def predict(Xs):
        return predict_model(model, Xs, emb, ens, mu, sd)

    best, best_state, best_ep = np.inf, None, -1
    for ep in range(epochs):
        perm = torch.randperm(len(Za), generator=g).to(dev)
        for i in range(0, len(Za), bs):
            idx = perm[i:i + bs]
            out = model(Xa[idx])
            if ens:
                loss = (Wa[idx, None] * (out - Ya[idx, None]) ** 2).mean()
            else:
                loss = (Wa[idx] * (out - Ya[idx]) ** 2).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sched.step()
        if fixed_ep is not None:
            if ep == fixed_ep:
                if save_path:
                    torch.save({"state_dict": model.state_dict(), "mu": mu, "sd": sd, "cont": cont,
                                "arch": dict(d_in=Za.shape[1], hidden=tuple(hidden), drop=drop, emb=emb, ens=ens,
                                             n_out=ya.shape[1] if multi else 1)}, save_path + ".pt")
                return predict(Xb), predict(Xt), ep
            continue
        pb = predict(Xb)
        score = np.sqrt(np.average((pb - yb) ** 2, weights=wbn))
        if score < best:
            best, best_ep = score, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return predict(Xb), predict(Xt), best_ep


def te_block(d, a, b, windows, cells=False, knn=False):
    """Per-fold spatial encodings (+ FB07 kNN target encodings), standardised with fitting-part statistics."""
    Fa = d.te_features(a, "oof", windows, cells)
    Fb = d.te_features(a, d.te_apply(b), windows, cells)
    Ft = d.te_features(a, d.te_apply(), windows, cells)
    if knn:
        Fa = pd.concat([Fa, d.knn_features(a, "oof")], axis=1)
        Fb = pd.concat([Fb, d.knn_features(a, ("train", b))], axis=1)
        Ft = pd.concat([Ft, d.knn_features(a, "test")], axis=1)
    cnt = [c for c in Fa.columns if c.endswith("_cnt")]
    for F in (Fa, Fb, Ft):
        F[cnt] = np.log1p(F[cnt])
    mu, sd = Fa.mean(), Fa.std().replace(0, 1)
    out = []
    for F in (Fa, Fb, Ft):
        Z = ((F - mu) / sd).clip(-6, 6)
        nanf = F.isna().astype("float32").add_suffix("_nan")
        out.append(np.hstack([Z.fillna(0).values, nanf.values]).astype("float32"))
    return out


def run_nn(d, name, kind, out_dir="exp", wmul=None, use_te=False, windows=(), seeds=(SEED,), cells=False,
           vary_order=False, extra=None, extra_t=None, model_dir=None, knn=False, **kw):
    """vary_order: mini-batch order seeded per (seed, fold) instead of one shared order.
    extra / extra_t: additional feature frames for train / test rows (e.g. Data.agg_features())."""
    set_seed()
    if extra is None:
        Z, Zt = preprocess(d.X, d.Xt)
    else:
        Z, Zt = preprocess(pd.concat([d.X.reset_index(drop=True), extra], axis=1),
                           pd.concat([d.Xt.reset_index(drop=True), extra_t], axis=1))
    y, w, mult = target_spec(d, kind)
    mult_t = target_spec(d, kind, test=True)[2]
    if wmul is not None:
        w = wmul / wmul.mean() if w is None else w * wmul / np.mean(w * wmul)
    oof = np.zeros(len(Z))
    pt = np.zeros(len(Zt))
    t0 = time.time()
    eps = []
    for k, (a, b) in enumerate(d.folds):
        Za, Zb, Ztk = Z[a], Z[b], Zt
        if use_te:
            Ta, Tb, Tt = te_block(d, a, b, windows, cells, knn)
            Za, Zb, Ztk = np.hstack([Za, Ta]), np.hstack([Zb, Tb]), np.hstack([Zt, Tt])
        for s in seeds:
            set_seed(s + k)
            if model_dir:
                os.makedirs(model_dir, exist_ok=True)
            pb, ptk, ep = fit_fold(Za, y[a], None if w is None else w[a], Zb, y[b],
                                   None if w is None else w[b], Ztk,
                                   save_path=f"{model_dir}/fold{k}_seed{s}" if model_dir else None,
                                   **(dict(kw, order_seed=1000 * s + k) if vary_order else kw))
            if is_log_kind(kind):
                pb, ptk = np.exp(pb), np.exp(ptk)
            oof[b] += pb * (mult if np.isscalar(mult) else mult[b]) / len(seeds)
            pt += ptk * mult_t / len(d.folds) / len(seeds)
            eps.append(ep)
        print(f"  [{name}] fold {k} best_ep={eps[-len(seeds):]} {time.time() - t0:.0f}s", flush=True)
    np.save(f"{out_dir}/oof/{name}.npy", oof)
    np.save(f"{out_dir}/pred/{name}.npy", pt)
    if is_gas_kind(kind):
        msg = (f"eui={nrmse(d.y[TARGET_COLS[2]], oof, TARGET_COLS[2]):.5f} "
               f"gas={nrmse(d.y[TARGET_COLS[0]], oof * d.A, TARGET_COLS[0]):.5f}")
    else:
        msg = f"elec={nrmse(d.y[TARGET_COLS[1]], oof, TARGET_COLS[1]):.5f}"
    print(f"[{name}] {msg} epochs={eps} time={time.time() - t0:.0f}s", flush=True)
    return oof, pt


def run_nn_multi(d, name, tasks, out_dir="exp", use_te=False, windows=(), seeds=(SEED,), cells=False, extra=None,
                 extra_t=None, model_dir=None, knn=False, **kw):
    """Multi-task MLP: one network fits several target formulations at once (shared trunk, one output each).
    tasks: list of (suffix, kind, wmul); each task's weights are normalised to mean 1, so the tasks count equally.
    Saves one OOF / test prediction per task as <name>_<suffix> (same physical scales as run_nn)."""
    set_seed()
    if extra is None:
        Z, Zt = preprocess(d.X, d.Xt)
    else:
        Z, Zt = preprocess(pd.concat([d.X.reset_index(drop=True), extra], axis=1),
                           pd.concat([d.Xt.reset_index(drop=True), extra_t], axis=1))
    Ys, Ws, mults, mults_t = [], [], [], []
    for _, kind, wmul in tasks:
        y, w, mult = target_spec(d, kind)
        if wmul is not None:
            w = wmul if w is None else w * wmul
        Ys.append(y)
        Ws.append(np.ones(len(y)) if w is None else w / np.mean(w))
        mults.append(mult)
        mults_t.append(target_spec(d, kind, test=True)[2])
    Y, W = np.column_stack(Ys), np.column_stack(Ws)
    oof = np.zeros((len(Z), len(tasks)))
    pt = np.zeros((len(Zt), len(tasks)))
    t0 = time.time()
    for k, (a, b) in enumerate(d.folds):
        Za, Zb, Ztk = Z[a], Z[b], Zt
        if use_te:
            Ta, Tb, Tt = te_block(d, a, b, windows, cells, knn)
            Za, Zb, Ztk = np.hstack([Za, Ta]), np.hstack([Zb, Tb]), np.hstack([Zt, Tt])
        for s in seeds:
            set_seed(s + k)
            if model_dir:
                os.makedirs(model_dir, exist_ok=True)
            pb, ptk, _ = fit_fold(Za, Y[a], W[a], Zb, Y[b], W[b], Ztk,
                                  save_path=f"{model_dir}/fold{k}_seed{s}" if model_dir else None, **kw)
            for j, (_, kind, _) in enumerate(tasks):
                qb, qt = (np.exp(pb[:, j]), np.exp(ptk[:, j])) if is_log_kind(kind) else (pb[:, j], ptk[:, j])
                m = mults[j]
                oof[b, j] += qb * (m if np.isscalar(m) else m[b]) / len(seeds)
                pt[:, j] += qt * mults_t[j] / len(d.folds) / len(seeds)
        print(f"  [{name}] fold {k} {time.time() - t0:.0f}s", flush=True)
    msg = []
    for j, (suffix, kind, _) in enumerate(tasks):
        np.save(f"{out_dir}/oof/{name}_{suffix}.npy", oof[:, j])
        np.save(f"{out_dir}/pred/{name}_{suffix}.npy", pt[:, j])
        if is_gas_kind(kind):
            msg.append(f"{suffix}: eui={nrmse(d.y[TARGET_COLS[2]], oof[:, j], TARGET_COLS[2]):.5f} "
                       f"gas={nrmse(d.y[TARGET_COLS[0]], oof[:, j] * d.A, TARGET_COLS[0]):.5f}")
        else:
            msg.append(f"{suffix}: elec={nrmse(d.y[TARGET_COLS[1]], oof[:, j], TARGET_COLS[1]):.5f}")
    print(f"[{name}] {' | '.join(msg)} time={time.time() - t0:.0f}s", flush=True)
    return oof, pt



In [ ]:
%%writefile dicefest_nb/src/blend.py
"""Non-negative least-squares blending of saved OOF predictions, one weight vector per scored column."""
import numpy as np
from scipy.optimize import nnls


def nnls_blend(P, y, row_scale=None):
    """Weights w >= 0 minimising ||s*(y - P w)|| where s is an optional per-row scale (e.g. floor area)."""
    if row_scale is None:
        row_scale = np.ones(len(y))
    w, _ = nnls(P * row_scale[:, None], y * row_scale)
    return w


def cv_blend_score(P, y, row_scale=None, n_splits=5, seed=0):
    """Honest blend score: weights fitted on 4/5 of rows, scored on the remaining 1/5."""
    from sklearn.model_selection import KFold
    s = np.ones(len(y)) if row_scale is None else row_scale
    pred = np.zeros(len(y))
    for a, b in KFold(n_splits, shuffle=True, random_state=seed).split(P):
        w = nnls_blend(P[a], y[a], s[a])
        pred[b] = P[b] @ w
    return pred


In [ ]:
%%writefile dicefest_nb/src/adopt_v2.py
"""Electric-heating adoption, v2.

Census structure: in every subzone (census output area) the hvac_dist_* shares are exact counts / N, where N is the
number of households. The smallest N that makes every share integral recovers N exactly. The households outside the
listed categories number K = (100 - sum(hvac_dist_*)) * N / 100, and they split into electric heating (the target)
and one missing category r: adoption = 100 * (K - count_r) / N.

Rows of subzones present in train copy the train value (the target is constant within a subzone). For the ~8% of
test rows in unseen subzones, count_r is predicted at subzone level by averaging two LightGBM models:
- q_xent: share q = count_r / K, binomial cross-entropy with weight K;
- r_reg: r = 100 * count_r / N, squared error.
Features: the subzone's census columns, means of the postcode-level columns over its train + test rows, N, K, and
pooled count_r / K of neighbouring labelled subzones (zone, climate cell, sub_n windows).

Validation is 5-fold over train subzones, also reweighted to the row-count mix of the unseen test subzones (mostly
single-row subzones with large K). Unseen-weighted RMSE: old model 1.042, q_xent 1.033, r_reg 1.035, average 1.030.

"""
import os
import sys

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

sys.path.insert(0, os.path.dirname(__file__))
from common import HVAC, SEED, base_features, constant_cols, load  # noqa: E402

BASE = dict(learning_rate=0.03, num_leaves=31, min_data_in_leaf=50, feature_fraction=0.5, bagging_fraction=0.8,
            bagging_freq=1, lambda_l2=5.0, seed=SEED, deterministic=True, force_row_wise=True, num_threads=2,
            verbose=-1)
WINDOWS = (1, 3, 10, 30)
# row counts of the test rows in unseen subzones, by the subzone's total row count (6 = 6 or more)
UNSEEN_MIX = {1: 5916, 2: 5158, 3: 2664, 4: 944, 5: 255, 6: 42}


def household_count(shares, n_max=1000, tol=2e-4):
    """Smallest N <= n_max with every share * N / 100 within tol of an integer (shares: (n, k) percentages)."""
    V = np.asarray(shares, dtype=float) / 100.0
    best = np.zeros(len(V), int)
    for N in range(1, n_max + 1):
        ok = (best == 0) & (np.abs(V * N - np.round(V * N)) < tol * max(1, N / 100)).all(1)
        best[ok] = N
    return best


def subzone_table(tr, te):
    const = constant_cols(tr)
    X = pd.concat([base_features(tr, const), base_features(te, const)], ignore_index=True)
    al = pd.concat([tr, te], ignore_index=True)
    X["subzone_id"] = al["subzone_id"].values
    nun = X.iloc[:len(tr)].groupby("subzone_id").nunique().max()
    oa = [c for c in X.columns if c != "subzone_id" and nun[c] <= 1 and c != "sub_n"]
    pc = [c for c in X.columns if c != "subzone_id" and nun[c] > 1]
    g = X.groupby("subzone_id", sort=False)
    S = g[oa].first().join(g[pc].mean().add_suffix("_m"))
    S["nrow"] = g.size()
    S["nrow_tr"] = tr.groupby("subzone_id").size().reindex(S.index).fillna(0).values
    S["sub_n"] = g["sub_n"].first().astype(int)
    S["zone"] = al.groupby("subzone_id", sort=False)["zone_region_id"].first().reindex(S.index).values
    cell = al.groupby(["env_climate_01", "env_climate_03", "env_climate_05"], sort=False).ngroup()
    S["cell"] = cell.groupby(al["subzone_id"]).first().reindex(S.index).values
    S["N"] = household_count(S[HVAC].values)
    S["rest"] = 100 - S[HVAC].sum(axis=1)
    S["K"] = np.round(S.rest * S.N / 100).astype(int)
    for h in HVAC:
        S[h + "_cnt"] = np.round(S[h] * S.N / 100)
    ad = tr.groupby("subzone_id", sort=False)["target_electric_heating_adoption"].first()
    S["ad"] = ad.reindex(S.index).values
    S["cr"] = np.round(S.K - S.ad * S.N / 100)  # NaN for unlabelled subzones
    return S


def neighbour_feats(F, A, size):
    """Pooled count_r / K of fit subzones F around apply subzones A (zone, climate cell, sub_n windows)."""
    out = {}
    for key in ("zone", "cell"):
        sK, sC, n = F.groupby(key).K.sum(), F.groupby(key).cr.sum(), F.groupby(key).size()
        k, c = A[key].map(sK).fillna(0).values, A[key].map(sC).fillna(0).values
        out[f"nb_{key}_q"] = np.where(k > 0, c / np.maximum(k, 1), np.nan)
        out[f"nb_{key}_K"] = k
        out[f"nb_{key}_n"] = A[key].map(n).fillna(0).values
    pos = F.sub_n.values
    cK = np.r_[0, np.cumsum(np.bincount(pos, F.K.values, size))]
    cC = np.r_[0, np.cumsum(np.bincount(pos, F.cr.values, size))]
    cN = np.r_[0, np.cumsum(np.bincount(pos, None, size))]
    for w in WINDOWS:
        lo, hi = np.clip(A.sub_n.values - w, 0, size), np.clip(A.sub_n.values + w + 1, 0, size)
        k, c = cK[hi] - cK[lo], cC[hi] - cC[lo]
        out[f"w{w}_q"] = np.where(k > 0, c / np.maximum(k, 1), np.nan)
        out[f"w{w}_K"] = k
        out[f"w{w}_n"] = cN[hi] - cN[lo]
    return pd.DataFrame(out, index=A.index)


def fit_predict(Fa, La, Fb, Lb, Fs, Ls, mode):
    """Train one model on (Fa, La) with early stopping on (Fb, Lb); return predicted r for Lb and Ls rows."""
    if mode == "q_xent":
        ma, mb = La.K.values > 0, Lb.K.values > 0
        qa, qb = (La.cr / La.K.clip(lower=1)).values, (Lb.cr / Lb.K.clip(lower=1)).values
        m = lgb.train(dict(BASE, objective="cross_entropy"), lgb.Dataset(Fa[ma], qa[ma], weight=La.K.values[ma]),
                      5000, valid_sets=[lgb.Dataset(Fb[mb], qb[mb], weight=Lb.K.values[mb])],
                      callbacks=[lgb.early_stopping(200, verbose=False)])
        to_r = lambda F, L: 100 * L.K.values * np.clip(m.predict(F, num_iteration=m.best_iteration), 0, 1) / L.N.values  # noqa: E731
    else:
        m = lgb.train(dict(BASE, objective="regression"), lgb.Dataset(Fa, (La.rest - La.ad).values), 5000,
                      valid_sets=[lgb.Dataset(Fb, (Lb.rest - Lb.ad).values)],
                      callbacks=[lgb.early_stopping(200, verbose=False)])
        to_r = lambda F, L: np.clip(m.predict(F, num_iteration=m.best_iteration), 0, None)  # noqa: E731
    return to_r(Fb, Lb), to_r(Fs, Ls), m.best_iteration


def cv_scores(L, r_oof):
    e2 = (np.clip(L.rest.values - r_oof, 0, 100) - L.ad.values) ** 2
    byn = pd.Series(e2).groupby(np.minimum(L.nrow.values, 6)).mean()
    return np.sqrt(e2.mean()), np.sqrt(sum(UNSEEN_MIX[k] * byn[k] for k in UNSEEN_MIX) / sum(UNSEEN_MIX.values()))


def fit_adoption_v2(cache="cache", verbose=True):
    tr, te = load(cache)
    S = subzone_table(tr, te)
    lab = S.ad.notna().values
    L, U = S[lab], S[~lab]
    size = int(S.sub_n.max()) + 2 * max(WINDOWS) + 2
    feats = [c for c in S.columns if c not in ("zone", "cell", "ad", "cr")]
    modes = ("q_xent", "r_reg")
    oof = {m: np.zeros(len(L)) for m in modes}
    pu = {m: np.zeros(len(U)) for m in modes}
    folds = list(KFold(5, shuffle=True, random_state=SEED).split(L))
    for k, (a, b) in enumerate(folds):
        La, Lb = L.iloc[a], L.iloc[b]
        inner = KFold(5, shuffle=True, random_state=SEED).split(La)
        na = pd.concat([neighbour_feats(La.iloc[ia], La.iloc[ib], size) for ia, ib in inner]).reindex(La.index)
        Fa = La[feats].join(na)
        Fb = Lb[feats].join(neighbour_feats(La, Lb, size))
        Fu = U[feats].join(neighbour_feats(La, U, size))
        for mode in modes:
            rb, ru, it = fit_predict(Fa, La, Fb, Lb, Fu, U, mode)
            oof[mode][b] = rb
            pu[mode] += ru / len(folds)
            if verbose:
                print(f"  adoption fold {k} {mode} it={it}", flush=True)
    r_oof = np.mean([oof[m] for m in modes], axis=0)
    r_u = np.mean([pu[m] for m in modes], axis=0)
    if verbose:
        for m in modes:
            print(f"{m}: CV rmse all {cv_scores(L, oof[m])[0]:.4f} unseen-weighted {cv_scores(L, oof[m])[1]:.4f}")
        s_all, s_un = cv_scores(L, r_oof)
        print(f"average: CV rmse all {s_all:.4f} unseen-weighted {s_un:.4f}", flush=True)
    known = dict(zip(L.index, L.ad.values))
    unseen = dict(zip(U.index, np.clip(U.rest.values - r_u, 0, 100)))
    pred = np.array([known[s] if s in known else unseen[s] for s in te["subzone_id"].values])
    frac_unseen = float(np.mean([s not in known for s in te["subzone_id"].values]))
    return pred, cv_scores(L, r_oof)[1], frac_unseen



In [ ]:
%%writefile dicefest_nb/src/dup_fix.py
"""Copy train targets to test rows that duplicate a train row (the same postcode record listed twice).

Match key: socio_demog_005 (near-unique), subzone_id, bld_dim_12, bld_dim_01, bld_dim_06. In train, all 92 groups of
rows sharing this key have identical values for all five targets. 77 test rows match a train row.

"""
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(__file__))
from common import TARGET_COLS, load  # noqa: E402

KEY = ["socio_demog_005", "subzone_id", "bld_dim_12", "bld_dim_01", "bld_dim_06"]


def duplicate_targets(tr, te):
    """Train targets for the test rows whose key matches a train row (NaN elsewhere), in test row order."""
    grp = tr.groupby(KEY, sort=False)[TARGET_COLS]
    assert (grp.nunique() == 1).all().all(), "train duplicates disagree on a target"
    ref = grp.first().reset_index()
    out = te[KEY].merge(ref, on=KEY, how="left")
    assert len(out) == len(te)
    return out[TARGET_COLS].values



In [ ]:
%%writefile dicefest_nb/src/postprocess.py
"""Post-processing of the blend: group + similarity-kNN residual correction (frozen weights), copied verbatim from
src/resid_corr.py and src/resid_knn.py (development versions, which also contain the holdout fit of the weights).

The correction of a test row is a linear combination (weights lambda in configs/resid_knn.json, fitted on an internal
70/30 holdout of train) of shrunken means of the blend's OOF residuals of train rows that share its zone, climate cell
or +-30 sub_n window, and of the k most similar train rows (building-stock feature space) within a +-W sub_n window.
"""
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

from blend import nnls_blend

K = 30    # shrinkage n / (n + K) of a group mean
WIN = 30  # half-width of the sub_n window
KSHR = 5.0  # shrinkage n / (n + KSHR) of a kNN mean
CONFIGS = {
    "gas": [("full", 100, 20, 4.0), ("era", 30, 20, 0.0), ("fc", 300, 50, 0.0), ("full", 10, 10, 0.0),
            ("era", 300, 50, 0.0), ("full", 3000, 200, 0.0)],
    "eui": [("full", 30, 10, 0.0), ("fc", 300, 50, 0.0), ("full", 3000, 200, 0.0), ("era", 30, 20, 0.0),
            ("full", 10, 10, 0.0), ("small", 30, 20, 0.0)],
    "elec": [("full", 100, 20, 4.0), ("full", 1000, 100, 0.0), ("full", 30, 20, 0.0), ("era", 300, 50, 0.0),
             ("small", 30, 20, 0.0)],
}


def group_keys(tr, te, subn, subnt):
    """(zone, climate cell, sub_n) codes for train and test rows"""
    both = pd.concat([tr, te])
    zone = pd.factorize(both["zone_region_id"])[0]
    cell = pd.factorize(pd.Series(list(zip(both["env_climate_01"], both["env_climate_03"], both["env_climate_05"]))))[0]
    n = len(tr)
    return (zone[:n], cell[:n], subn), (zone[n:], cell[n:], subnt)


def group_mean(gf, ga, num, den):
    """n/(n+K) * sum(num)/sum(den) over fit rows in the apply row's group"""
    ng = max(gf.max(), ga.max()) + 1
    s, dn, n = (np.bincount(gf, v, ng)[ga] for v in (num, den, np.ones(len(gf))))
    return n / (n + K) * np.where(dn > 0, s / np.where(dn > 0, dn, 1), 0.0)


def window_mean(sf, sa, num, den):
    """as group_mean, over fit rows with |sub_n - sub_n(apply row)| <= WIN"""
    m = max(sf.max(), sa.max()) + 1
    lo, hi = np.clip(sa - WIN, 0, m), np.clip(sa + WIN + 1, 0, m)
    s, dn, n = ((c := np.r_[0, np.cumsum(np.bincount(sf, v, m))])[hi] - c[lo] for v in (num, den, np.ones(len(sf))))
    return n / (n + K) * np.where(dn > 0, s / np.where(dn > 0, dn, 1), 0.0)


def features(metric, kf, ka, r, p_fit, p_app, A_fit, A_app):
    """Correction features of apply rows from fit-row residuals r on the metric's scale.
    gas: per-area shift times the row's area; eui: area-weighted mean residual (EUI noise variance ~ 1/area);
    elec: relative shift times the row's prediction."""
    if metric == "gas":
        num, den, mult = r, A_fit, A_app
    elif metric == "eui":
        num, den, mult = A_fit * r, A_fit, 1.0
    else:
        num, den, mult = r, p_fit, p_app
    return np.column_stack([group_mean(kf[0], ka[0], num, den), group_mean(kf[1], ka[1], num, den),
                            window_mean(kf[2], ka[2], num, den)]) * np.reshape(mult, (-1, 1))


def blend_preds(Pf, Pa, yf, metric, A_fit, A_app):
    """NNLS blend as in make_submission.py; returns predictions on the metric's scale for fit and apply rows"""
    if metric == "gas":
        w = nnls_blend(Pf, yf, A_fit)
        return Pf @ w * A_fit, Pa @ w * A_app
    w = nnls_blend(Pf, yf)
    return Pf @ w, Pa @ w


def spaces(tr, te):
    """similarity spaces for train + test rows, standardized with train statistics"""
    both = pd.concat([tr, te])
    n = len(tr)
    A, M, E = both["bld_dim_06"].values, both["bld_dim_12"].values.astype(float), both["bld_dim_01"].values.astype(float)
    era = both[[f"bld_era_{i:02d}" for i in range(9, 16)]].values / 100

    def std(Z):
        return (Z - Z[:n].mean(0)) / Z[:n].std(0)

    full = std(np.column_stack([np.log(A / M), np.log(E / M), np.log(M), era, both["bld_dim_02"].values / E,
                                np.log1p(both["bld_dim_17"].values), np.log(both["bld_dim_03"].values / A)]))
    soc_cols = [c for c in tr.columns if c.startswith(("socio_demog", "hvac_dist"))
                and pd.api.types.is_numeric_dtype(tr[c]) and tr[c].nunique() > 1]
    soc = std(both[soc_cols].values)
    pca = PCA(8, random_state=0).fit(soc[:n])
    cen = std(pca.transform(soc))
    return {"full": full, "era": std(np.column_stack([np.log(A / M), era])),
            "small": std(np.column_stack([np.log(A / M), np.log(E / M), np.log(M)])),
            "cen": cen, "fc": np.column_stack([full, cen * 0.5])}


def knn_mean(Zf, sf, Za, sa, num, den, W, k, spatial=0.0):
    """sum(num)/sum(den) over the k fit rows nearest to each apply row in Z (+ spatial * |sub_n diff| / W) among
    fit rows with |sub_n diff| <= W, shrunk by n/(n+KSHR)"""
    o = np.argsort(sf, kind="stable")
    so = sf[o]
    lo, hi = np.searchsorted(so, sa - W, "left"), np.searchsorted(so, sa + W, "right")
    L = int((hi - lo).max())
    chunk = max(256, int(2e7 // (L * (Zf.shape[1] + 1))))
    numo, deno, Zo, sof = num[o], den[o], Zf[o], so.astype(float)
    out, cnt = np.zeros(len(sa)), np.zeros(len(sa))
    for s in range(0, len(sa), chunk):
        idx = np.arange(s, min(s + chunk, len(sa)))
        valid = np.arange(L)[None, :] < (hi[idx] - lo[idx])[:, None]
        cand = np.where(valid, lo[idx, None] + np.arange(L)[None, :], 0)
        dist = ((Zo[cand] - Za[idx][:, None, :]) ** 2).sum(-1)
        if spatial:
            dist = dist + spatial * np.abs(sof[cand] - sa[idx, None]) / max(W, 1)
        dist = np.where(valid, dist, np.inf)
        kk = min(k, L)
        part = np.argpartition(dist, kk - 1, axis=1)[:, :kk]
        sel, ok = np.take_along_axis(cand, part, 1), np.isfinite(np.take_along_axis(dist, part, 1))
        sn, sd = (numo[sel] * ok).sum(1), (deno[sel] * ok).sum(1)
        out[idx] = np.where(sd > 0, sn / np.where(sd > 0, sd, 1), 0.0)
        cnt[idx] = ok.sum(1)
    return cnt / (cnt + KSHR) * out


def all_features(metric, kf, ka, r, p_fit, p_app, A_fit, A_app, Zs, f_rows, a_rows, sub_f, sub_a):
    """group features of resid_corr.py followed by the metric's kNN features (same metric scaling)"""
    F = features(metric, kf, ka, r, p_fit, p_app, A_fit, A_app)
    if metric == "gas":
        num, den, mult = r, A_fit, A_app
    elif metric == "eui":
        num, den, mult = A_fit * r, A_fit, 1.0
    else:
        num, den, mult = r, p_fit, p_app
    cols = [knn_mean(Zs[space][f_rows], sub_f, Zs[space][a_rows], sub_a, num, den, W, k, sp) * mult
            for space, W, k, sp in CONFIGS[metric]]
    return np.column_stack([F] + cols)


In [ ]:
%%writefile dicefest_nb/src/pipeline.py
"""Final DICE FEST 2026 pipeline: data -> 50 cross-validated models -> NNLS blend -> adoption model -> residual
correction -> train-duplicate copy -> submission.

Stages (all deterministic, seeds fixed in the configs):
1. members  : every model of configs/members.yaml, 5- or 10-fold KFold(shuffle, random_state=42) over train rows,
              saving OOF and fold-averaged test predictions to <work>/oof|pred/<name>.npy. Two lanes run in parallel:
              'gpu' (XGBoost CUDA, PyTorch MLPs) and 'cpu' (LightGBM).
2. adoption : electric-heating adoption (exact household counts, LightGBM for unseen subzones), adopt_v2.py.
3. assemble : non-negative least-squares blend per scored column on the OOF predictions, clean-building ratio from
              its exact identity, residual correction with frozen weights (configs/resid_knn.json), copy of the train
              targets for the 77 test rows that duplicate a train row, clip, write CSV + sha256.
"""
import hashlib
import json
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import KFold

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
from common import SEED, SIGMAS, TARGET_COLS, load, load_sample, nrmse  # noqa: E402

CONF = os.path.join(HERE, "..", "configs")
ELEC = ("epm", "elec", "lepm", "epa", "lepa")
GAS, EL, EUI = TARGET_COLS[0], TARGET_COLS[1], TARGET_COLS[2]
METRICS = {"gas": GAS, "eui": EUI, "elec": EL}


def load_config():
    return yaml.safe_load(open(os.path.join(CONF, "members.yaml")))


def _wmul(d, code):
    return {None: None, "iA": 1.0 / d.A, "iM": 1.0 / d.M, "A": d.A, "M": d.M.astype(float),
            "sqrtM": np.sqrt(d.M)}[code]


LANES = ("xgb", "nn", "cpu")  # parallel worker processes: XGBoost (GPU), MLPs (GPU), LightGBM (CPU)
LANE_THREADS = {"xgb": "2", "nn": "2", "cpu": "12", "gpu": "4"}


def lane_of(spec, lane):
    """does the member belong to `lane`? ('gpu' = xgb + nn, kept for a two-lane run)"""
    b = spec["backend"]
    kind = "xgb" if b == "xgb" else ("nn" if b in ("nn", "nn_multi") else "cpu")
    return kind == lane or (lane == "gpu" and kind in ("xgb", "nn"))


def run_members(lane, work, only=None):
    """train every member of a lane (in config order); members with existing outputs are skipped"""
    from nn import run_nn, run_nn_multi
    from train_lib import Data, run
    cfg = load_config()
    os.makedirs(f"{work}/oof", exist_ok=True)
    os.makedirs(f"{work}/pred", exist_ok=True)
    d = Data()
    d.knn_cache_dir = f"{work}/knn_cache"  # FB07 fold encodings shared between members and lanes (same values)
    ex, ext = d.agg_features()
    done_multi = set()
    for name, spec in cfg["members"].items():
        if not lane_of(spec, lane) or (only and name not in only):
            continue
        if spec["backend"] == "nn_multi":
            g = spec["group"]
            if g in done_multi:
                continue
            done_multi.add(g)
            m = cfg["multi"][g]
            outs = [f"{work}/pred/{g}_{t[0]}.npy" for t in m["tasks"]]
            if all(os.path.exists(o) for o in outs):
                continue
            d.folds = list(KFold(m["n_folds"], shuffle=True, random_state=SEED).split(d.X))
            tasks = [(s, k, _wmul(d, w)) for s, k, w in m["tasks"]]
            agg = dict(extra=ex, extra_t=ext) if m["agg"] else {}
            run_nn_multi(d, g, tasks, out_dir=work, use_te=m["use_te"], windows=tuple(m["windows"]), cells=m["cells"],
                         knn=m.get("knn", False), epochs=m["epochs"], fixed_ep=m["fixed_ep"], seeds=tuple(m["seeds"]),
                         model_dir=f"{work}/models/{g}", **agg, **m["params"])
            continue
        if os.path.exists(f"{work}/pred/{name}.npy"):
            continue
        d.folds = list(KFold(spec["n_folds"], shuffle=True, random_state=SEED).split(d.X))
        agg = dict(extra=ex, extra_t=ext) if spec["agg"] else {}
        te = dict(use_te=spec.get("use_te", False), windows=tuple(spec.get("windows", ())),
                  cells=spec.get("cells", False), knn=spec.get("knn", False))
        if spec["backend"] == "nn":
            p = dict(spec["params"])
            p["hidden"] = tuple(p["hidden"])
            run_nn(d, name, spec["kind"], out_dir=work, wmul=_wmul(d, spec["wmul"]), epochs=spec["epochs"],
                   fixed_ep=spec["fixed_ep"], seeds=tuple(spec["seeds"]), model_dir=f"{work}/models/{name}", **te,
                   **agg, **p)
        else:
            run(d, name, spec["backend"], spec["kind"], dict(spec["params"]), out_dir=work,
                wmul=_wmul(d, spec["wmul"]), model_dir=f"{work}/models/{name}", **te, **agg)


def run_adoption(work):
    from adopt_v2 import fit_adoption_v2
    pred, rmse_unseen, frac_unseen = fit_adoption_v2()
    np.save(f"{work}/pred/adopt_v2.npy", pred)
    return {"adoption_unseen_weighted_rmse": rmse_unseen, "unseen_fraction": frac_unseen}


def _correction(args):
    """residual-correction features of one metric (run in a worker process)"""
    from postprocess import all_features
    metric, kt, ke, r, pf, pa, A, At, Zs, n, n_te, subn, subnt, lam = args
    F = all_features(metric, kt, ke, r, pf, pa, A, At, Zs, np.arange(n), np.arange(n, n + n_te), subn, subnt)
    return metric, F @ np.asarray(lam)


def assemble(work, out_csv, log=print):
    from blend import cv_blend_score, nnls_blend
    from dup_fix import duplicate_targets
    from postprocess import KSHR, CONFIGS, blend_preds, group_keys, spaces
    cfg = load_config()
    tr, te = load()
    A, At = tr["bld_dim_06"].values, te["bld_dim_06"].values
    y = {c: tr[c].values for c in TARGET_COLS}
    runs = {"gas": cfg["blend"]["gas"], "elec": cfg["blend"]["elec"]}
    P = {f: (np.column_stack([np.load(f"{work}/oof/{r}.npy") for r in runs[f]]),
             np.column_stack([np.load(f"{work}/pred/{r}.npy") for r in runs[f]])) for f in runs}
    summary = {}
    # 1. blend (as src/make_submission.py) and its honest nested-CV score
    w = {"eui": nnls_blend(P["gas"][0], y[EUI]), "gas": nnls_blend(P["gas"][0], y[EUI], row_scale=A),
         "elec": nnls_blend(P["elec"][0], y[EL])}
    summary["weights"] = {m: dict(zip(runs["elec" if m == "elec" else "gas"], np.round(w[m], 6).tolist())) for m in w}
    summary["cv_blend_honest"] = {
        "gas": nrmse(y[GAS], cv_blend_score(P["gas"][0], y[EUI], A) * A, GAS),
        "eui": nrmse(y[EUI], cv_blend_score(P["gas"][0], y[EUI]), EUI),
        "elec": nrmse(y[EL], cv_blend_score(P["elec"][0], y[EL]), EL)}
    log(f"blend honest CV {summary['cv_blend_honest']}")
    sub = pd.DataFrame({"id": te["id"].values})
    sub[GAS] = np.clip(P["gas"][1] @ w["gas"], 0, None) * At
    sub[EL] = np.clip(P["elec"][1] @ w["elec"], 0, None)
    sub[EUI] = np.clip(P["gas"][1] @ w["eui"], 0, None)
    sub[TARGET_COLS[3]] = np.round(100 * te["bld_dim_02"].values / te["bld_dim_01"].values, 2)
    sub[TARGET_COLS[4]] = np.clip(np.load(f"{work}/pred/adopt_v2.npy"), 0, 100)
    # 2. residual correction (group + similarity kNN), weights frozen from the holdout fit
    lamcfg = json.load(open(os.path.join(CONF, "resid_knn.json")))
    assert lamcfg["configs"] == {m: [list(c) for c in v] for m, v in CONFIGS.items()} and lamcfg["kshr"] == KSHR
    subn = tr["subzone_id"].str[4:].astype(int).values
    subnt = te["subzone_id"].str[4:].astype(int).values
    kt, ke = group_keys(tr, te, subn, subnt)
    Zs = spaces(tr, te)
    jobs, base = [], {}
    for metric, col in METRICS.items():
        fam = "elec" if metric == "elec" else "gas"
        yv = y[EL if metric == "elec" else EUI]
        pf, pa = blend_preds(P[fam][0], P[fam][1], yv, metric, A, At)
        assert np.allclose(np.clip(pa, 0, None), sub[col].values, rtol=1e-9, atol=1e-9)
        r = (A if metric == "gas" else 1.0) * yv - pf
        base[metric] = pa
        jobs.append((metric, kt, ke, r, pf, pa, A, At, Zs, len(tr), len(te), subn, subnt, lamcfg["lam"][metric]))
    with ProcessPoolExecutor(3) as ex:
        for metric, corr in ex.map(_correction, jobs):
            col = METRICS[metric]
            sub[col] = np.clip(base[metric] + corr, 0, None)
            summary[f"correction_rms_over_sigma_{metric}"] = float(np.sqrt(np.mean(corr ** 2)) / SIGMAS[col])
            log(f"{metric} correction rms/sigma {summary[f'correction_rms_over_sigma_{metric}']:.5f}")
    # 3. test rows identical to a train row get its targets
    dup = duplicate_targets(tr, te)
    hit = ~np.isnan(dup[:, 0])
    for j, col in enumerate(TARGET_COLS):
        sub.loc[hit, col] = dup[hit, j]
    summary["duplicate_rows_copied"] = int(hit.sum())
    # 4. checks and output
    ss = load_sample()
    assert list(sub.columns) == list(ss.columns) and (sub["id"].values == ss["id"].values).all()
    assert np.isfinite(sub[TARGET_COLS].values).all()
    os.makedirs(os.path.dirname(os.path.abspath(out_csv)), exist_ok=True)
    sub.to_csv(out_csv, index=False)
    sha = hashlib.sha256(open(out_csv, "rb").read()).hexdigest()
    summary["sha256"] = sha
    json.dump(summary, open(f"{work}/summary.json", "w"), indent=1)
    log(f"wrote {out_csv} sha256 {sha}")
    return sha


In [ ]:
%%writefile dicefest_nb/src/inference.py
"""Re-create every member's OOF and test predictions from the saved fold models (no training).

Mirrors train_lib.run / nn.run_nn / nn.run_nn_multi step by step: the same features (target encodings of each fold's
fitting part, aggregates), the same prediction calls (XGBoost inplace_predict on CUDA up to the best iteration,
LightGBM saved at its best iteration, MLP eval-mode prediction in the same chunks), the same back-transforms.
Used by inference_from_artifacts.ipynb; check_member() compares the result with the arrays of a training run.
"""
import os

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import KFold

from common import SEED
from pipeline import load_config


def _gbm_predict(backend, path, Xb, Xt):
    if backend == "xgb":
        import xgboost as xgb
        m = xgb.Booster()
        m.load_model(path + ".ubj")
        m.set_param({"device": "cuda"})
        rng = (0, m.best_iteration + 1)
        return m.inplace_predict(Xb, iteration_range=rng), m.inplace_predict(Xt, iteration_range=rng)
    import lightgbm as lgb
    m = lgb.Booster(model_file=path + ".txt")
    return m.predict(Xb), m.predict(Xt)


def predict_gbm(d, name, spec, model_dir, extra=None, extra_t=None):
    from train_lib import is_log_kind, target_spec
    kind = spec["kind"]
    _, _, mult = target_spec(d, kind)
    mult_t = target_spec(d, kind, test=True)[2]
    use_te, windows, cells = spec.get("use_te", False), tuple(spec.get("windows", ())), spec.get("cells", False)
    oof, pt = np.zeros(len(d.X)), np.zeros(len(d.Xt))
    for k, (a, b) in enumerate(d.folds):
        Xb, Xt = d.X.iloc[b], d.Xt
        if extra is not None:
            Xb = pd.concat([Xb.reset_index(drop=True), extra.iloc[b].reset_index(drop=True)], axis=1)
            Xt = pd.concat([Xt.reset_index(drop=True), extra_t.reset_index(drop=True)], axis=1)
        if use_te:
            Xb = pd.concat([Xb.reset_index(drop=True), d.te_features(a, d.te_apply(b), windows, cells)], axis=1)
            Xt = pd.concat([Xt.reset_index(drop=True), d.te_features(a, d.te_apply(), windows, cells)], axis=1)
        if spec.get("knn"):
            Xb = pd.concat([Xb.reset_index(drop=True), d.knn_features(a, ("train", b))], axis=1)
            Xt = pd.concat([Xt.reset_index(drop=True), d.knn_features(a, "test")], axis=1)
        pb, ptk = _gbm_predict(spec["backend"], f"{model_dir}/fold{k}", Xb, Xt)
        if is_log_kind(kind):
            pb, ptk = np.exp(pb), np.exp(ptk)
        oof[b] = pb * (mult if np.isscalar(mult) else mult[b])
        pt += ptk * mult_t / len(d.folds)
    return oof, pt


def _load_mlp(path):
    from nn import MLP
    ck = torch.load(path + ".pt", map_location="cuda", weights_only=False)
    ar = ck["arch"]
    model = MLP(ar["d_in"], ar["hidden"], ar["drop"], ar["emb"], ck["cont"], ar["ens"], n_out=ar["n_out"]).to("cuda")
    model.load_state_dict(ck["state_dict"])
    return model, ar, ck["mu"], ck["sd"]


def _nn_inputs(d, extra, extra_t):
    from nn import preprocess
    if extra is None:
        return preprocess(d.X, d.Xt)
    return preprocess(pd.concat([d.X.reset_index(drop=True), extra], axis=1),
                      pd.concat([d.Xt.reset_index(drop=True), extra_t], axis=1))


def predict_nn(d, name, spec, model_dir, extra=None, extra_t=None, tasks=None):
    """single-task (spec['kind']) or multi-task (tasks = [(suffix, kind, wmul code)]) MLP"""
    from nn import predict_model, set_seed, te_block
    from train_lib import is_log_kind, target_spec
    set_seed()
    Z, Zt = _nn_inputs(d, extra, extra_t)
    kinds = [spec["kind"]] if tasks is None else [t[1] for t in tasks]
    mults = [target_spec(d, k)[2] for k in kinds]
    mults_t = [target_spec(d, k, test=True)[2] for k in kinds]
    seeds = spec["seeds"]
    oof, pt = np.zeros((len(Z), len(kinds))), np.zeros((len(Zt), len(kinds)))
    for k, (a, b) in enumerate(d.folds):
        Zb, Ztk = Z[b], Zt
        if spec.get("use_te"):
            _, Tb, Tt = te_block(d, a, b, tuple(spec["windows"]), spec["cells"], spec.get("knn", False))
            Zb, Ztk = np.hstack([Zb, Tb]), np.hstack([Zt, Tt])
        Xb, Xt = torch.tensor(Zb, device="cuda"), torch.tensor(Ztk, device="cuda")
        for s in seeds:
            model, ar, mu, sd = _load_mlp(f"{model_dir}/fold{k}_seed{s}")
            pb, ptk = predict_model(model, Xb, ar["emb"], ar["ens"], mu, sd), predict_model(model, Xt, ar["emb"],
                                                                                           ar["ens"], mu, sd)
            pb, ptk = pb.reshape(len(b), -1), ptk.reshape(len(Zt), -1)
            for j, kind in enumerate(kinds):
                qb, qt = (np.exp(pb[:, j]), np.exp(ptk[:, j])) if is_log_kind(kind) else (pb[:, j], ptk[:, j])
                m = mults[j]
                oof[b, j] += qb * (m if np.isscalar(m) else m[b]) / len(seeds)
                pt[:, j] += qt * mults_t[j] / len(d.folds) / len(seeds)
    return oof, pt


def predict_all(work_models, out_dir, only=None):
    """writes <out_dir>/oof|pred/<member>.npy for every member (or `only`) from <work_models>/<member>/"""
    from train_lib import Data
    cfg = load_config()
    os.makedirs(f"{out_dir}/oof", exist_ok=True)
    os.makedirs(f"{out_dir}/pred", exist_ok=True)
    d = Data()
    d.knn_cache_dir = f"{out_dir}/knn_cache"
    ex, ext = d.agg_features()
    done = set()
    for name, spec in cfg["members"].items():
        if only and name not in only:
            continue
        if spec["backend"] == "nn_multi":
            g = spec["group"]
            if g in done:
                continue
            done.add(g)
            m = cfg["multi"][g]
            d.folds = list(KFold(m["n_folds"], shuffle=True, random_state=SEED).split(d.X))
            agg = (ex, ext) if m["agg"] else (None, None)
            oof, pt = predict_nn(d, g, m, f"{work_models}/{g}", *agg, tasks=m["tasks"])
            for j, t in enumerate(m["tasks"]):
                np.save(f"{out_dir}/oof/{g}_{t[0]}.npy", oof[:, j])
                np.save(f"{out_dir}/pred/{g}_{t[0]}.npy", pt[:, j])
            continue
        d.folds = list(KFold(spec["n_folds"], shuffle=True, random_state=SEED).split(d.X))
        agg = (ex, ext) if spec["agg"] else (None, None)
        if spec["backend"] == "nn":
            oof, pt = predict_nn(d, name, spec, f"{work_models}/{name}", *agg)
            oof, pt = oof[:, 0], pt[:, 0]
        else:
            oof, pt = predict_gbm(d, name, spec, f"{work_models}/{name}", *agg)
        np.save(f"{out_dir}/oof/{name}.npy", oof)
        np.save(f"{out_dir}/pred/{name}.npy", pt)
        print(f"[{name}] predicted from saved models", flush=True)


def check_member(out_dir, ref_dir, name):
    res = {}
    for part in ("oof", "pred"):
        a, b = np.load(f"{out_dir}/{part}/{name}.npy"), np.load(f"{ref_dir}/{part}/{name}.npy")
        res[part] = "identical" if np.array_equal(a, b) else f"max abs diff {np.abs(a - b).max():.3g}"
    return res


In [ ]:
%%writefile dicefest_nb/src/metric.py
"""Official competition metric (verbatim) plus a per-target helper."""
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

TARGET_COLS = [
    'target_macro_gas_kwh',
    'target_macro_elec_kwh',
    'target_eui_gas_kwh_per_sqm',
    'target_clean_building_ratio',
    'target_electric_heating_adoption'
]

OFFICIAL_SIGMAS = {
    'target_macro_gas_kwh': 151750.04,
    'target_macro_elec_kwh': 41725.37,
    'target_eui_gas_kwh_per_sqm': 40.65,
    'target_clean_building_ratio': 12.49,
    'target_electric_heating_adoption': 7.18
}


def compute_column_wise_nrmse(y_true, y_pred, sigmas=OFFICIAL_SIGMAS):
    y_true_vals = y_true[TARGET_COLS].values if isinstance(y_true, pd.DataFrame) else np.array(y_true)
    y_pred_vals = y_pred[TARGET_COLS].values if isinstance(y_pred, pd.DataFrame) else np.array(y_pred)
    nrmse_list = []
    for idx, col in enumerate(TARGET_COLS):
        rmse = np.sqrt(mean_squared_error(y_true_vals[:, idx], y_pred_vals[:, idx]))
        nrmse_list.append(rmse / sigmas[col])
    return float(np.mean(nrmse_list))


def nrmse_per_target(y_true, y_pred, targets=TARGET_COLS, sigmas=OFFICIAL_SIGMAS):
    """dict target -> RMSE / sigma for any subset of targets; inputs are DataFrames or dicts of arrays."""
    return {t: float(np.sqrt(np.mean((np.asarray(y_true[t], float) - np.asarray(y_pred[t], float)) ** 2)) / sigmas[t])
            for t in targets}


In [ ]:
%%writefile dicefest_nb/src/seed.py
"""Seed locking, called at the start of every process before CUDA is used."""
import os
import random

import numpy as np


def seed_everything(seed: int) -> None:
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # must be set before CUDA is initialised
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True)
    except ImportError:
        pass


In [ ]:
%%writefile dicefest_nb/configs/members.yaml
members:
  xk_gpm:
    backend: xgb
    kind: gpm
    params: {learning_rate: 0.03, max_depth: 10, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: iA
    n_folds: 10
    agg: true
    source: src/exp_v16_knn.py
    use_te: true
    windows: [2, 10]
    cells: true
    knn: true
  xk_lgpm:
    backend: xgb
    kind: lgas_per:bld_dim_12
    params: {learning_rate: 0.03, max_depth: 10, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: A
    n_folds: 10
    agg: true
    source: src/exp_v16_knn.py
    use_te: true
    windows: [2, 10]
    cells: true
    knn: true
  xk_gpgea:
    backend: xgb
    kind: gas_per:bld_dim_03
    params: {learning_rate: 0.03, max_depth: 10, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: iA
    n_folds: 10
    agg: true
    source: src/exp_v16_knn.py
    use_te: true
    windows: [2, 10]
    cells: true
    knn: true
  xk_lepa:
    backend: xgb
    kind: lelec_per:bld_dim_06
    params: {learning_rate: 0.05, max_depth: 8, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: M
    n_folds: 10
    agg: true
    source: src/exp_v16_knn.py
    use_te: true
    windows: [2, 10]
    cells: true
    knn: true
  lk_gpm:
    backend: lgb
    kind: gpm
    params: {learning_rate: 0.04, num_leaves: 255, min_data_in_leaf: 100, feature_fraction: 0.5, bagging_fraction: 0.8, bagging_freq: 1, lambda_l2: 10.0,
      max_bin: 255, num_threads: 12}
    wmul: iA
    n_folds: 10
    agg: true
    source: src/exp_v16_knn.py
    use_te: true
    windows: [2, 10]
    cells: true
    knn: true
  lk_lepa:
    backend: lgb
    kind: lelec_per:bld_dim_06
    params: {learning_rate: 0.04, num_leaves: 255, min_data_in_leaf: 100, feature_fraction: 0.5, bagging_fraction: 0.8, bagging_freq: 1, lambda_l2: 10.0,
      max_bin: 255, num_threads: 12}
    wmul: M
    n_folds: 10
    agg: true
    source: src/exp_v16_knn.py
    use_te: true
    windows: [2, 10]
    cells: true
    knn: true
  pk_gpgea:
    backend: nn
    kind: gas_per:bld_dim_03
    epochs: 12
    fixed_ep: 11
    wmul: iA
    params:
      lr: 0.001
      wd: 0.0001
      drop: 0.3
      hidden: [1024, 512, 256]
      emb: {k: 24, sigma: 0.05, d: 24}
    seeds: [42, 43, 44]
    n_folds: 10
    agg: true
    source: src/exp_v16_knn.py
    use_te: true
    windows: [2, 10]
    cells: true
    knn: true
  pk_lgpm:
    backend: nn
    kind: lgas_per:bld_dim_12
    epochs: 12
    fixed_ep: 11
    wmul: A
    params:
      lr: 0.001
      wd: 0.0001
      drop: 0.3
      hidden: [1024, 512, 256]
      emb: {k: 24, sigma: 0.05, d: 24}
    seeds: [42, 43, 44]
    n_folds: 10
    agg: true
    source: src/exp_v16_knn.py
    use_te: true
    windows: [2, 10]
    cells: true
    knn: true
  pk_lepm:
    backend: nn
    kind: lepm
    epochs: 10
    fixed_ep: 9
    wmul: M
    params:
      lr: 0.001
      wd: 0.0001
      drop: 0.3
      hidden: [1024, 512, 256]
      emb: {k: 24, sigma: 0.02, d: 48}
    seeds: [42, 43, 44]
    n_folds: 10
    agg: true
    source: src/exp_v16_knn.py
    use_te: true
    windows: [2, 10]
    cells: true
    knn: true
  mk1_gpm: {backend: nn_multi, group: mk1, source: src/exp_v16_knn.py}
  mk1_epm: {backend: nn_multi, group: mk1, source: src/exp_v16_knn.py}
  xa_gpm:
    backend: xgb
    kind: gpm
    params: {learning_rate: 0.03, max_depth: 10, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: iA
    n_folds: 10
    agg: true
    source: src/exp_v10_agg.py
    use_te: true
    windows: [2, 10]
    cells: true
  xa_gpgea:
    backend: xgb
    kind: gas_per:bld_dim_03
    params: {learning_rate: 0.03, max_depth: 10, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: iA
    n_folds: 10
    agg: true
    source: src/exp_v10_agg.py
    use_te: true
    windows: [2, 10]
    cells: true
  p1_gpm:
    backend: nn
    kind: gpm
    epochs: 44
    fixed_ep: 43
    wmul: iA
    params:
      lr: 0.001
      wd: 0.0001
      drop: 0.3
      hidden: [1024, 512, 256]
      emb: {k: 24, sigma: 0.05, d: 24}
    seeds: [42, 43, 44]
    n_folds: 10
    agg: false
    use_te: true
    windows: [2, 10]
    cells: true
    source: src/exp_v9_nn.py
  lgb_gas:
    backend: lgb
    kind: gas
    params: {learning_rate: 0.04, num_leaves: 255, min_data_in_leaf: 100, feature_fraction: 0.5, bagging_fraction: 0.8, bagging_freq: 1, lambda_l2: 10.0,
      max_bin: 255, num_threads: 12}
    wmul: null
    n_folds: 5
    agg: false
    source: src/exp_lgb1.py
  x8_gpn:
    backend: xgb
    kind: gas_per:bld_dim_01
    params: {learning_rate: 0.05, max_depth: 10, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: iA
    n_folds: 10
    agg: false
    source: src/exp_v8_xgb.py
    use_te: true
    windows: [2, 10]
    cells: true
  n6_lgpm:
    backend: nn
    kind: lgas_per:bld_dim_12
    epochs: 40
    fixed_ep: 39
    wmul: A
    params:
      lr: 0.001
      wd: 0.0001
      drop: 0.3
      hidden: [1024, 512, 256]
    seeds: [42, 43, 44]
    n_folds: 10
    agg: false
    use_te: true
    windows: [2, 10]
    cells: true
    source: src/exp_v6_nn.py
  xgb_eui_wsM:
    backend: xgb
    kind: eui
    params: {learning_rate: 0.05, max_depth: 8, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: sqrtM
    n_folds: 5
    agg: false
    source: src/exp_zoo1.py
  la_gas:
    backend: lgb
    kind: gas
    params: {learning_rate: 0.04, num_leaves: 255, min_data_in_leaf: 100, feature_fraction: 0.5, bagging_fraction: 0.8, bagging_freq: 1, lambda_l2: 10.0,
      max_bin: 255, num_threads: 12}
    wmul: null
    n_folds: 10
    agg: true
    source: src/exp_v10_lgb.py
    use_te: true
    windows: [2, 10]
    cells: true
  xgb_leui_wA:
    backend: xgb
    kind: leui
    params: {learning_rate: 0.05, max_depth: 8, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: A
    n_folds: 5
    agg: false
    source: src/exp_zoo2.py
  xgb_gpn_tew:
    backend: xgb
    kind: gas_per:bld_dim_01
    params: {learning_rate: 0.05, max_depth: 8, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: iA
    n_folds: 5
    agg: false
    source: src/exp_zoo5.py
    use_te: true
    windows: [2, 10]
    cells: false
  la_leui_wA:
    backend: lgb
    kind: leui
    params: {learning_rate: 0.04, num_leaves: 255, min_data_in_leaf: 100, feature_fraction: 0.5, bagging_fraction: 0.8, bagging_freq: 1, lambda_l2: 10.0,
      max_bin: 255, num_threads: 12}
    wmul: A
    n_folds: 10
    agg: true
    source: src/exp_v10_lgb.py
    use_te: true
    windows: [2, 10]
    cells: true
  n4f_gpm:
    backend: nn
    kind: gpm
    epochs: 60
    fixed_ep: 37
    wmul: iA
    params:
      lr: 0.001
      wd: 0.0001
      drop: 0.3
      hidden: [1024, 512, 256]
    seeds: [42, 43, 44]
    n_folds: 10
    agg: false
    use_te: true
    windows: [2, 10]
    cells: true
    source: src/exp_nnfix.py
  x8_lgpm:
    backend: xgb
    kind: lgas_per:bld_dim_12
    params: {learning_rate: 0.03, max_depth: 10, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: A
    n_folds: 10
    agg: false
    source: src/exp_v8_xgb.py
    use_te: true
    windows: [2, 10]
    cells: true
  xgb_gas:
    backend: xgb
    kind: gas
    params: {learning_rate: 0.05, max_depth: 8, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: null
    n_folds: 5
    agg: false
    source: src/exp_zoo1.py
  x8_gpm:
    backend: xgb
    kind: gpm
    params: {learning_rate: 0.03, max_depth: 10, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: iA
    n_folds: 10
    agg: false
    source: src/exp_v8_xgb.py
    use_te: true
    windows: [2, 10]
    cells: true
  xgb_lepm:
    backend: xgb
    kind: lepm
    params: {learning_rate: 0.05, max_depth: 8, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: M
    n_folds: 5
    agg: false
    source: src/exp_zoo2.py
  t1_lepm:
    backend: nn
    kind: lepm
    epochs: 24
    fixed_ep: 23
    wmul: M
    params:
      lr: 0.002
      wd: 0.0001
      drop: 0.1
      hidden: [512, 512, 512]
      emb: {k: 24, sigma: 0.05, d: 24}
      ens: 16
    seeds: [42]
    n_folds: 10
    agg: true
    use_te: true
    windows: [2, 10]
    cells: true
    source: src/exp_v11_nn.py
  l4_epa:
    backend: lgb
    kind: elec_per:bld_dim_06
    params: {learning_rate: 0.04, num_leaves: 255, min_data_in_leaf: 100, feature_fraction: 0.5, bagging_fraction: 0.8, bagging_freq: 1, lambda_l2: 10.0,
      max_bin: 255, num_threads: 12}
    wmul: iM
    n_folds: 10
    agg: false
    source: src/exp_v5_lgb.py
    use_te: true
    windows: [2, 10]
    cells: true
  la_lepa:
    backend: lgb
    kind: lelec_per:bld_dim_06
    params: {learning_rate: 0.04, num_leaves: 255, min_data_in_leaf: 100, feature_fraction: 0.5, bagging_fraction: 0.8, bagging_freq: 1, lambda_l2: 10.0,
      max_bin: 255, num_threads: 12}
    wmul: M
    n_folds: 10
    agg: true
    source: src/exp_v10_lgb.py
    use_te: true
    windows: [2, 10]
    cells: true
  n6b_lepm:
    backend: nn
    kind: lepm
    epochs: 24
    fixed_ep: 23
    wmul: M
    params:
      lr: 0.001
      wd: 0.0001
      drop: 0.3
      hidden: [1024, 512, 256]
    seeds: [45, 46, 47]
    n_folds: 10
    agg: false
    use_te: true
    windows: [2, 10]
    cells: true
    source: src/exp_v6b_nn.py
  x4_elec:
    backend: xgb
    kind: elec
    params: {learning_rate: 0.05, max_depth: 8, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: null
    n_folds: 10
    agg: false
    source: src/exp_v4_xgb.py
    use_te: true
    windows: [2, 10]
    cells: true
  xgb_epm_wM_tew_k10:
    backend: xgb
    kind: epm
    params: {learning_rate: 0.05, max_depth: 8, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: iM
    n_folds: 10
    agg: false
    source: src/exp_k10.py
    use_te: true
    windows: [2, 10]
    cells: false
  xgb_elec:
    backend: xgb
    kind: elec
    params: {learning_rate: 0.05, max_depth: 8, subsample: 0.8, colsample_bytree: 0.6, min_child_weight: 50, reg_lambda: 1.0, eval_metric: rmse}
    wmul: null
    n_folds: 5
    agg: false
    source: src/exp_zoo1.py
multi:
  mk1:
    tasks:
    - [gpm, gpm, iA]
    - [epm, epm, iM]
    epochs: 12
    fixed_ep: 11
    params:
      lr: 0.001
      wd: 0.0001
      drop: 0.3
      hidden: [2048, 1024, 512]
      emb: {k: 24, sigma: 0.05, d: 24}
    seeds: [42, 43, 44]
    n_folds: 10
    agg: true
    source: src/exp_v16_knn.py
    use_te: true
    windows: [2, 10]
    cells: true
    knn: true
blend:
  gas: [xa_gpm, xa_gpgea, p1_gpm, lgb_gas, x8_gpn, n6_lgpm, xgb_eui_wsM, la_gas, xgb_leui_wA, xgb_gpn_tew, la_leui_wA, n4f_gpm, x8_lgpm, xgb_gas, x8_gpm,
    xk_gpm, mk1_gpm, xk_gpgea, xk_lgpm, pk_gpgea, lk_gpm, pk_lgpm]
  elec: [xgb_lepm, t1_lepm, l4_epa, la_lepa, n6b_lepm, x4_elec, xgb_epm_wM_tew_k10, xgb_elec, mk1_epm, pk_lepm, xk_lepa, lk_lepa]


In [ ]:
%%writefile dicefest_nb/configs/resid_knn.json
{
 "lam": {
  "gas": [
   0.42824691423854605,
   0.34324321599743735,
   -0.04507533930417333,
   0.026098390669680855,
   0.1299011514492503,
   0.08839194808124867,
   0.1190698580713099,
   0.12895078938223772,
   0.17143729403739827
  ],
  "eui": [
   0.5768212474196011,
   0.3272988973273137,
   -0.0361505231892388,
   0.05327733614750403,
   0.11847764191685797,
   0.28785980981517917,
   0.11798688546401045,
   0.0930297059133312,
   0.04950354804988893
  ],
  "elec": [
   0.3439831096021096,
   0.015640909939605863,
   0.130350825157795,
   0.004282142530294261,
   0.11063543156905072,
   0.04782358078728994,
   0.07905721282727439,
   0.08557832209601494
  ]
 },
 "configs": {
  "gas": [
   [
    "full",
    100,
    20,
    4.0
   ],
   [
    "era",
    30,
    20,
    0.0
   ],
   [
    "fc",
    300,
    50,
    0.0
   ],
   [
    "full",
    10,
    10,
    0.0
   ],
   [
    "era",
    300,
    50,
    0.0
   ],
   [
    "full",
    3000,
    200,
    0.0
   ]
  ],
  "eui": [
   [
    "full",
    30,
    10,
    0.0
   ],
   [
    "fc",
    300,
    50,
    0.0
   ],
   [
    "full",
    3000,
    200,
    0.0
   ],
   [
    "era",
    30,
    20,
    0.0
   ],
   [
    "full",
    10,
    10,
    0.0
   ],
   [
    "small",
    30,
    20,
    0.0
   ]
  ],
  "elec": [
   [
    "full",
    100,
    20,
    4.0
   ],
   [
    "full",
    1000,
    100,
    0.0
   ],
   [
    "full",
    30,
    20,
    0.0
   ],
   [
    "era",
    300,
    50,
    0.0
   ],
   [
    "small",
    30,
    20,
    0.0
   ]
  ]
 },
 "kshr": 5.0
}

In [ ]:
sys.path.insert(0, os.path.join(ROOT, "src"))
import pipeline
cfg = pipeline.load_config()
print(len(cfg["members"]), "blend members")

## Verifikasi checksum artefak

In [ ]:
bad = 0
for line in open(os.path.join(ARTIFACTS, "SHA256SUMS")):
    h, rel = line.split(maxsplit=1)
    p = os.path.join(ARTIFACTS, rel.strip())
    if hashlib.sha256(open(p, "rb").read()).hexdigest() != h:
        bad += 1
        print("checksum mismatch", rel)
assert bad == 0
print("all artifact checksums ok")

## Prediksi dari model tersimpan

In [ ]:
import inference
os.makedirs(WORK, exist_ok=True)
inference.predict_all(os.path.join(ARTIFACTS, "models"), WORK)

## Adopsi, blend, koreksi, submisi

In [ ]:
info = pipeline.run_adoption(WORK)
sha = pipeline.assemble(WORK, OUT_CSV)
print("sha256", sha)